# **Reinforcement Learning Agent For CNN Hyper Parameter Tunning**

## **Let create base CNN model - Modified LeNet5 CNN**

#### **torch**

- This is the main PyTorch library.
- Provides :
   - Tensors (like NumPy arrays but can run on GPU)
   - Automatic differentiation (autograd) for gradients
   - Neural network modules (nn), optimizers, loss functions
   - GPU acceleration with CUDA
   - Essentially, it’s the core engine for building and running deep learning models.

#### **torchvision**

- This is an official PyTorch library for computer vision tasks.
- Provides:
- Datasets: e.g., CIFAR-10, MNIST, ImageNet
- Transforms: for preprocessing images (resize, crop, normalize, convert to tensors)
- Pretrained models: e.g., ResNet, VGG, AlexNet

#### **matplotlib** 
- graph plotting (loss curves, accuracy graphs)

#### NOTE : 
  - **torchvision** must match to installed **torch** version.

### IMPORT LIBRARIES
- no need to import every wher same vlibrary but when work with .py we have to do

### Pkg installation

In [1]:
# RUN ON GPU
# conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia

# RUN ON CPU
# conda install pytorch torchvision cpuonly -c pytorch # only cpu pytorch matplotlib

In [2]:
import torch # main deep learning library
import torch.nn as nn # contains neural network layers and loss functions
import torch.optim as optim # optimization algorithms (SGD, Adam, etc.)

import torchvision # datasets + transforms for images
import torchvision.transforms as transforms

import matplotlib

print(torch.__version__)
print(torchvision.__version__)
print(matplotlib.__version__)

2.5.1
0.20.1
3.10.9


### DEVICE CONFIGURATION

In [3]:
# If GPU is available → use it
# Otherwise → fallback to CPU
     
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device : ", device)

Using device :  cpu


### What is a Tensor?

A tensor is just a generalization of arrays to more dimensions. In PyTorch (and machine learning in general), tensors are the basic data structure for holding numbers.

Think of it as: cube having h x d x l x ...

Tensor Types

0D tensor	x = torch.tensor(5)	A single number (scalar)<br>
1D tensor	x = torch.tensor([1,2,3])	A vector (like a list of numbers)<br>
2D tensor	x = torch.tensor([[1,2],[3,4]])	A matrix (rows × columns)<br>
3D tensor	Shape (channels, height, width)	An image (for grayscale or RGB channels)<br>
4D tensor	Shape (batch_size, channels, height, width)	A batch of images

## **DATA PREPROCESSING**

#### Transforms
- Transforms are applied to each image before feeding into neural network use later when you do have images

In [4]:
transform = transforms.Compose([
    # Converts PIL Image or NumPy array (pixel values 0–255) to a PyTorch tensor, Also scales pixel values from 0–255 → 0–1 (float)
    transforms.ToTensor(), 
    
    # Standardizes the tensor values to make training more stable and help the network converge faster, normalized_pixel=(pixel - mean)/std
    # This scales the pixel range from 0–1 → -1–1, which is often better for neural networks with activation functions like tanh or ReLU.
    transforms.Normalize(mean=(0.5,), std=(0.5,))
])

### Data set downloading
- Download MNIST training dataset

- **Note** : PyTorch creates a dataset object, It just remembers your transform function, Transforms run only when you access an image


In [5]:
trainset = torchvision.datasets.MNIST(
    root="./data", # folder to store dataset
    train=True,  # training data
    download=True, # download if not exists
    transform=transform # apply preprocessing
)

testset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

### Dataloaders
- DataLoader splits dataset into batches
- batch_size = 64
- Instead of sending the entire dataset into the model at once, it sends 64 samples per step.
- Example:
- If dataset size = 6400 → number of batches = 6400 / 64 = 100 batches
- Each training iteration processes one batch
- Why batching is important:
-    Reduces memory usage
-    Speeds up training (GPU optimized)
-    Provides more stable gradients than single-sample training
- During training, we usually set: shuffle=True
- During testing, we usually set: shuffle=False

In [6]:
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size = 64,
    shuffle=True
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=64,
    shuffle=False
)

### DEFINE MODIFIED LENET ARCHITECTURE

#### Classic LeNet-5 has:
- 2 conv layers
- 2 fully connected layers

#### My version adds:
- an extra convolution layer<br>
- more feature channels

##### nn.Module = base class for all neural networks
##### **How ModifiedLeNet class is working**
- inherit class from nn.Module which is the base class for all neural networks in PyTorch
- why we inherit?
   - it just say, My class (ModifiedLeNet) is a specialized version of nn.Module
   - our model automatically gets all features of nn.Module
   - What gain by inheriting from nn.Module?
      - Parameter tracking > model.parameters() > Optimizer needs this to update weights
      - GPU / device support > You can move model easily > model.to(device)
      - Saving and loading > torch.save(model.state_dict())
      - Training / evaluation mode > model.train() > model.eval()
      - Automatic layer registration > self.conv = nn.Conv2d(...)
- The __init__ function is the constructor. It runs once when you create the model object (model = ModifiedLeNet())
- __init__ is where we, define layers, register parameters, build architecture
- PyTorch doesn’t “scan my code”, It only tracks objects that are assigned to: self.some_name = layer

In [7]:
class ModifiedLeNet(nn.Module):

    # constructor → define layers >>> batch(64 images at once) → conv → activation → pooling → fc → output
    def __init__(self):
        super().__init__()

        # ================= CONVOLUTIONAL BLOCK (FEATURE EXTRACTOR) ==================================

        # This part extracts image features
        self.conv_layers = nn.Sequential(
            # ------------- Conv layer 1 ---> INPUT > [batch, 1, 28, 28] -------------
            # input channels = 1 (grayscale) since we use MNIST, out_channels = number of filters
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3), # OUTPUT > [batch, 32, 26, 26] >>> batch = number of images processed at once, Batch size is defined in DataLoader
            # ReLU activation introduces non-linearity
            nn.ReLU(), # max(0, x)
            # MaxPool reduces image size by half
            nn.MaxPool2d(2), # [batch, 32, 13, 13]
    
            # ------------- Conv layer 2 ---> INPUT > [batch, 32, 13, 13] -------------
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3), # OUTPUT > [batch, 64, 11, 11]
            nn.ReLU(),
            nn.MaxPool2d(2), # [batch, 64, 5, 5]
    
            # ------------- Conv layer 3 (extra layer = modification) ---> INPUT > [batch, 64, 5, 5] -------------
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3), # OUTPUT > [batch, 128, 3, 3] # This is why you later use: 128 * 3 * 3
            nn.ReLU()
        )

        # ================= FULLY CONNECTED BLOCK (CLASSIFIER (Head))==================================

        # This part performs classification
        self.fc_layers = nn.Sequential(
            # ------------- Flatten layer ------------- 
            # converts 3D feature map → 1D vector, Flatten does NOT learn anything, It has no weights, It is just reshaping data
            nn.Flatten(), # [batch, 128, 3, 3] → [batch, 1152]
    
            # -------------Linear layer (dense layer) -------------
            # Compresses features into a compact representation, 128 feature maps * 3 * 3 size
            nn.Linear(128 * 3 * 3, 128),
    
            nn.ReLU(), # Adds non-linearity again.
    
            # ------------ Linear layer (Output layer) ------------
            # 10 neurons = digits 0–9
            nn.Linear(128, 10) # [batch, 10]
        )

    # forward pass defines data flow
    # input image > conv layers → feature maps > flatten > fully connected layers  > class scores
    '''
    This is the actual computation pipeline of the model, When we call > output = model(images), PyTorch internally does:model.forward(images)
    So this function is the execution path
    
    def forward(self, x): >>> x = input batch(shape : [batch, channels, height, width])
    x = self.conv_layers(x) >>> This sends the input through : Conv → ReLU → Pool → Conv → ReLU → Pool → Conv → ReLU, 
    So after this line > x = extracted feature maps
    x = self.fc_layers(x) >>> Now x goes into > Flatten → Linear → ReLU → Linear
    return x >>> Returns predictions > Example output for one image: [0.2, 1.1, 0.3, 5.4, 0.1, 0.0, 0.2, 0.1, 0.0, 0.2] 
    > Largest value index = predicted digit
    '''
    def forward(self, x):
        # pass input through conv layers
        x = self.conv_layers(x)
        # pass result through fully connected layers
        x = self.fc_layers(x)
        return x
         

#### Create model instance and moves it to the selected hardware (CPU or GPU).

In [13]:
model = ModifiedLeNet().to(device)

### LOSS FUNCTION

How far is your prediction from the correct answer?”<br>
CrossEntropyLoss > standard loss for classification<br>
How far is your prediction from the correct answer?”<br>
creates the loss function your model will use to measure how wrong its predictions are during training in PyTorch<br>

A loss function tells the model:<br>
lower loss = better predictions<br><br>
Higher loss = worse predictions

0.0   — Perfect prediction<br>
0.01  — Extremely good<br>
0.05  — Excellent<br>
0.1   — Very good<br>
0.2   — Good<br>
0.3   — Decent<br>
0.5   — Acceptable<br>
0.7   — Still learning<br>
1.0   — Weak predictions<br>
1.5   — Poor<br>
2.0   — Bad<br>
2.3   — Random guessing (for 10 classes like MNIST)<br>
3.0   — Very bad<br>
5.0   — Terrible (confidently wrong)<br>
5<    — Completely wrong predictions

In [10]:
criterion = nn.CrossEntropyLoss()

### OPTIMIZER

##### What an Optimizer Does
After each forward pass:

    1. Model predicts

    2. Loss is calculated

    3. Gradients are computed (backpropagation)

    4 .Optimizer updates weights

So the optimizer is the part that actually makes model learn.

In [20]:
# SGD = stochastic gradient descent optimizer
# sets up the optimizer that will update neural network’s weights during training

optimizer = optim.SGD(
    model.parameters(),  # model weights
    lr=0.01              # learning rate
)

### TRAINING LOOP

In [22]:
'''
Here the place model training happening. but we didn't train the model? then how do it prediction ? 
before training, the model can still produce predictions because : neural network initialized with random initial weights
so yes initila guessing may very wrong
but then we calculate loss how the model wrong >>> "Epoch [1/10], Step [100/938], Loss: 2.2807" very high loss
then we move to loss.backward() >>> computes gradients, which give the direction on how each weight should change to reduce the error
next we move to step optimizer.step() >>> uses above gradients to update the weights
final summary >>> predict > compare > compute error > backpropagate > update weights > repeat
'''

# already defined in above cells
# lr for optimizer = 0.01


epochs = 2  # number of rounds that model train on entire dataset

for epoch in range(epochs):

    # set model to training mode
    model.train()

    running_loss = 0

    # iterate over batches(64 in each one)
    # There are 938 loops inside ONE epoch
    # if mnist has 60,000 images and batch_size =64, then 60000 / 64 ≈ 938 batches
    # Each loop returns ONE batch, 
    # iteration (i = 0) >>> images → first 64 images, labels → labels of first 64 images
    # Take 64 images → predict → compute loss → update weights → move to next 64 → repeat
    for i, (images, labels) in enumerate(trainloader):
        # move data to GPU/CPU
        images, labels = images.to(device), labels.to(device)

        # reset gradients from previous step
        optimizer.zero_grad()

        # forward pass → prediction
        output = model(images)

        # calculate loss
        loss = criterion(output, labels)

        # backward pass > compute gradients
        loss.backward()

        # update weights
        optimizer.step()

        # accumulate loss
        running_loss += loss.item()

        # Loss - avg loss so far, for 100, 200, 300, ...
        if (i+1) % 200 == 0:  # print every 100 batches
            print(f"Epoch [{epoch+1}/{epochs}], batches [{i+1}/{len(trainloader)}] done, Loss: {loss.item():.4f}")


    # print average loss per epoch
    print(f"average loss for epoch {epoch+1}, Avg Loss: {running_loss/len(trainloader):.4f}\n")
    
print("Training done")

Epoch [1/2], batches [200/938] done, Loss: 0.2666
Epoch [1/2], batches [400/938] done, Loss: 0.0488
Epoch [1/2], batches [600/938] done, Loss: 0.0932
Epoch [1/2], batches [800/938] done, Loss: 0.0249
average loss for epoch 1, Avg Loss: 0.0831

Epoch [2/2], batches [200/938] done, Loss: 0.0359
Epoch [2/2], batches [400/938] done, Loss: 0.1317
Epoch [2/2], batches [600/938] done, Loss: 0.0923
Epoch [2/2], batches [800/938] done, Loss: 0.0237
average loss for epoch 2, Avg Loss: 0.0716

Training done


### TESTING PHASE

In [23]:
# switch to evaluation mode
model.eval()

correct = 0
total = 0

# no gradients needed during testing
with torch.no_grad():

    for images, labels in testloader:

        images, labels = images.to(device), labels.to(device)

        outputs = model(images)

        # choose class with highest probability
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()


# print final accuracy
print(f"\nFinal Test Accuracy: {100 * correct / total:.2f}%")


Final Test Accuracy: 98.15%


# **Q - Learning Algorithm with CNN**

- Before implementing Q-learning, first prepare the environment that Q-learning will control
- consider only one hyper parameter for now - **Learning Rate**
- Define RL Components

  **State** (What info agent sees)
  - state = current learning rate
  
  **Action** (What agent can do)
    - increase LR
    - decrease LR
    - keep LR sameincrease LR
    - decrease LR
    - keep LR same

  **Reward** (What agent tries to maximize)
    - reward = validation accuracy
 
  **Environment**
  - Environment = CNN training process

- Agent → chooses LR >>> 
CNN → trains one epoch >>> 
Environment → returns accuracy

- convert your CNN training loop into a function

### CREATE A FUNCTION TO TRAIN MODEL FOR Q-LEARNING ALGORITHM
- we can pass no of epochs, learning rate as hps

In [24]:

trainset = torchvision.datasets.MNIST(
    root="./data", # folder to store dataset
    train=True,  # training data
    download=True, # download if not exists
    transform=transform # apply preprocessing
)

testset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size = 64,
    shuffle=True
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=64,
    shuffle=False
)

In [25]:
import torch
import torch.nn as nn
import torch.optim as optim

# epochs - number of rounds that model train on entire dataset
# lr - lr of cnn optimizer

def train_model(device, model, lr, epochs):

    # loss function
    criterion = nn.CrossEntropyLoss()

    # Optimizer
    optimizer = optim.SGD(
        model.parameters(),  # model weights
        lr=lr              # learning rate
    )

    # Train model for 1 epoch only for testing
    for epoch in range(epochs):
        model.train()
        running_loss = 0

        # iterate over batches, When load MNIST using PyTorch dataset loaders, each batch gives: (images, labels) 64 sets(batches)
        # enumerate add index for each batch here we may have 938 batch all will be indexed as 1, 2, 3, ...
        for i, (images, labels) in enumerate(trainloader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() # summ all loss for each batch (938 losses), bcs we need entire dataset loss

            #if (i+1) % 100 == 0:  # print every 100 batches
            #    print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(trainloader)}], Loss: {loss.item():.4f}")

        # print average loss per epoch
        print(f"Epoch {epoch+1} Loss: {running_loss/len(trainloader):.4f}")
    
        model.eval()
        correct = 0
        total = 0
        
        # no gradients needed during testing
        with torch.no_grad():
            for images, labels in testloader:
                images, labels = images.to(device), labels.to(device)
                output = model(images)
                _, predicted = torch.max(output, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        # print final accuracy
        # print(f"\nFinal Test Accuracy: {100 * correct / total:.2f}%")
        accuracy = 100 * correct / total
        return accuracy

### Define Q-Learning Table

In [26]:
import numpy as np
import random

# Define possible learning rates (discretized)
# these are the "states"

learning_rates = [0.0001, 0.001, 0.01, 0.1]

# action that can taken by agent
# -1: decrease, 0: keep, 1: increase

actions = [-1, 0, 1] 

# Initialize Q-table: 
# rows = states, columns = actions
# len(learning_rates) → number of states
# Q-table keeps track of the expected reward for taking an action at a state
# at start Q-table is all zeros while someepisodes going it will fill(chekc later codes)

Q_table = np.zeros((len(learning_rates), len(actions)))

#### Hyperparameters for Q-learning

In [27]:
alpha = 0.1   
# learning rate > how fast Q values update
# small = slow learning, stable
# big = fast learning, unstable

gamma = 0.9   
# discount factor > how much the agent cares about future rewards
# 0   = only care about immediate reward
# 1   = care about long-term reward
# 0.9 = good balance

epsilon = 1.0 
# exploration rate > probability of taking random action
# start with full exploration (100% random)

epsilon_decay = 0.95
# after each episode, exploration reduces
# agent becomes less random and more intelligent

epsilon_min = 0.05
# minimum exploration → agent still explores 5% forever
# prevents getting stuck in bad strategy

max_steps = 2
# maximum steps per episode (how long agent can move before episode ends)

### Helper Functions for Q-Learning

In [28]:
# Map learning rate to state index ( return the index of passed lr value in the above deined lsit)
def get_state_index(lr):
    return learning_rates.index(lr)

# Return new learning rate based on action
# Ensures the learning rate stays within bounds with min(),max() 
# always increase/decrease by one step
def take_action(state_index, action):
    if action == -1: # decrease
        new_index = max(0, state_index - 1) # 0, 1, 2
    elif action == 0: # keep same
        new_index = state_index # 0, 1, 2, 3
    else: # increase
        new_index = min(len(learning_rates)-1, state_index+1) # 3, 2, 1
    return new_index, learning_rates[new_index]

### Q-Learning Training Loop

In [30]:
# Use proper max_steps=1 – 3, epcohs=3 – 8, episodes=20 – 50 values

'''
---EPISODE---
One complete run of an agent from start -> until it stops
Episode = one full experience / one game / one attempt
Start -> agent acts -> gets rewards -> reaches goal or fails > END
That whole journey = 1 episode

if cnn epoches = 10, RL episodes = 20, steps = 25 >>> in each RL episode CNN train 25 times, like wise 20 times RL do same thing, in each CNN training
time it will train 10 times on entire dataset

---STEPS---
how long agent can move before episode ends
in each step CNN will be trained , retuern acc, update q table

'''

episodes = 1  # how many trials the agent runs (20)
epochs= 1
max_steps= 2  # already define but for clarity

# Set table values to zero since i run several times this cell for testing
Q_table = np.zeros((len(learning_rates), len(actions)))

# Episodes × Steps = 5 × 10 = 50 times, The CNN training function is called 50 times, if cnn epochs =5,in each traing data set used 5 times(5x10x5)

for episode in range(episodes):

    print(f"Episode {episode+1} started with random hp values")
    
    # Start with random lr
    lr = random.choice(learning_rates)
    state_index = get_state_index(lr)

    # Initialize a new CNN model for each episode
    model = ModifiedLeNet().to(device)
    
    # Train model and get reward(acc)
    accuracy = train_model(
        device=device, 
        model=model, 
        lr=lr,
        epochs=epochs
    )
    
    print(f"randomly selected lr as {lr} > state_index = {state_index}, accuracy={accuracy}")

    done = False
    step = 0

    while not done:
        print(f"STEP {episode+1}.{step+1} started")

        model = ModifiedLeNet().to(device)
        
        # store previous accuracy
        old_accuracy = accuracy
        
        # Choose action: epsilon-greedy
        if random.uniform(0, 1) < epsilon:
            action_index = random.randint(0, len(actions)-1) # 0, 1, 2
            print(f"Explore action_index : {action_index} (0, 1, ,2)")
        else:
            # np.argmax([...]) returns the index of the maximum value in that Q_table[state_index] row
            # if all values are equal (all zeros), np.argmax returns index of the first occurrence of the maximum value(0)
            action_index = np.argmax(Q_table[state_index]) # exploit, range 0-2
            print(f"Exploit action_index : {action_index} (0, 1, ,2)")
            
        action = actions[action_index] # -1, 0, 1
        print(f"ACTION : {action} (-1, 0, 1)")
        
        # let pass current index(this is acatualy the state(lr) we pass its index instead value) & action choosen by above epsilon-greedy
        # then get new state index & learning rate
        new_state_index, new_lr = take_action(state_index, action) # state_index means current/ previous state's index
        print(f"new_state_index ={new_state_index} (old - {state_index})  , new_lr = {new_lr} (old - {lr})")

        # Train CNN with new learning rate
        new_accuracy = train_model(device, model, new_lr, epochs=epochs)

        new_reward = new_accuracy - old_accuracy
        print(f"new_accuracy ={new_accuracy} , new_reward = {new_reward}")

        # Update Q-table
        Q_value = Q_table[state_index, action_index] + alpha * (new_reward + gamma * np.max(Q_table[new_state_index])- Q_table[state_index, action_index])
        Q_table[state_index, action_index] = Q_value
        print(f"Q_value = {Q_value} added to ({state_index}, {action_index})")
        
        # move state forward
        state_index = new_state_index
        lr = new_lr
        accuracy = new_accuracy

        step += 1

        # Stop after 3 steps per episode to keep it simple
        if step >= max_steps:
            done = True

        # Prevent agent keeps exploring / exploit forever, decay epsilon
        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        print("NEW EPSILON : ", epsilon)
        # print("Q-table:\n", Q_table)
        print(f"Step {episode+1}.{step}/{max_steps} done\n")
    
    print(f"Episode {episode+1} DONE ( {max_steps} steps used), Final lr : {lr} & accuracy : {accuracy}, Reward: {new_reward:.2f}%")
    print("Q-table:\n", Q_table, "\n===============================\n")

print("================== DONE =====================")

Episode 1 started with random hp values
Epoch 1 Loss: 0.2973
randomly selected lr as 0.1 > state_index = 3, accuracy=96.77
STEP 1.1 started
Explore action_index : 1 (0, 1, ,2)
ACTION : 0 (-1, 0, 1)
new_state_index =3 (old - 3)  , new_lr = 0.1 (old - 0.1)
Epoch 1 Loss: 0.2955
new_accuracy =97.89 , new_reward = 1.1200000000000045
Q_value = 0.11200000000000046 added to (3, 1)
NEW EPSILON :  0.7737809374999999
Step 1.1/2 done

STEP 1.2 started
Explore action_index : 1 (0, 1, ,2)
ACTION : 0 (-1, 0, 1)
new_state_index =3 (old - 3)  , new_lr = 0.1 (old - 0.1)
Epoch 1 Loss: 0.2693
new_accuracy =98.46 , new_reward = 0.5699999999999932
Q_value = 0.16787999999999978 added to (3, 1)
NEW EPSILON :  0.7350918906249998
Step 1.2/2 done

Episode 1 DONE ( 2 steps used), Final lr : 0.1 & accuracy : 98.46, Reward: 0.57%
Q-table:
 [[0.      0.      0.     ]
 [0.      0.      0.     ]
 [0.      0.      0.     ]
 [0.      0.16788 0.     ]] 

================== DONE =====================


#### inspect Q-table

In [31]:
print("Q-table after training:")
for i, lr in enumerate(learning_rates):
    print(f"LR {lr}: {Q_table[i]}")

Q-table after training:
LR 0.0001: [0. 0. 0.]
LR 0.001: [0. 0. 0.]
LR 0.01: [0. 0. 0.]
LR 0.1: [0.      0.16788 0.     ]


### Select best learning rate

In [32]:
best_state = np.argmax(Q_table.max(axis=1))
best_lr = learning_rates[best_state]
print(f"Best learning rate found by Q-learning: {best_lr}")

Best learning rate found by Q-learning: 0.1


### Test model with new lr given byh RL Agent

In [ ]:
learning_rate = 0.01
rates = [0.0001, 0.001, 0.01, 0.1]
epochs = 1

for i in rates:
    model = ModifiedLeNet().to(device)
    epochs = 1
    acc = train_model(device=device,model=model, lr=i, epochs=epochs)
    print(f"Validation accuracy for lr={i}: {acc:.2f}%")